In [61]:
# !/usr/bin/env python3
"""
Event Data Aggregation Lab - Transform Data: SQL & Pandas Mastery
Course: Product Analytics Unlocked: From Metrics to Meaningful Insights

This lab focuses on applying data aggregation techniques to summarize event data
and identifying syntactic differences between SQL dialects for enterprise analytics.
"""

'\nEvent Data Aggregation Lab - Transform Data: SQL & Pandas Mastery\nCourse: Product Analytics Unlocked: From Metrics to Meaningful Insights\n\nThis lab focuses on applying data aggregation techniques to summarize event data\nand identifying syntactic differences between SQL dialects for enterprise analytics.\n'

In [62]:
import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [63]:
# PROVIDED CODE - DO NOT MODIFY
def create_sample_dataset():
    """Creates a sample user event dataset similar to Netflix viewer analytics data"""
    np.random.seed(42)  # For reproducible results

    # Generate sample user events
    n_events = 50000
    user_ids = np.random.randint(1000, 5000, n_events)
    device_types = np.random.choice(
        ['mobile', 'desktop', 'tablet', 'smart_tv'], n_events, p=[0.4, 0.3, 0.2, 0.1]
    )

    # Generate timestamps over the last 7 days
    base_time = datetime.now() - timedelta(days=7)
    time_offsets = np.random.randint(0, 7 * 24 * 60 * 60, n_events)  # seconds in 7 days
    timestamps = [base_time + timedelta(seconds=int(offset)) for offset in time_offsets]

    # Generate session data
    session_durations = np.random.exponential(15, n_events)  # minutes, exponential distribution
    action_types = np.random.choice(['view', 'click', 'search', 'purchase'], n_events, p=[0.5, 0.3, 0.15, 0.05])

    # Create DataFrame
    events_df = pd.DataFrame({
        'user_id': user_ids,
        'timestamp': timestamps,
        'device_type': device_types,
        'action_type': action_types,
        'session_duration_minutes': session_durations
    })

    # Sort by timestamp for realistic event ordering
    events_df = events_df.sort_values('timestamp').reset_index(drop=True)

    return events_df

In [64]:
def setup_sql_database(df):
    """Sets up SQLite database with sample data for SQL dialect comparison"""
    conn = sqlite3.connect(':memory:')
    df.to_sql('user_events', conn, index=False, if_exists='replace')
    return conn

In [65]:
# Initialize the dataset and database connection
print("Setting up sample dataset and database...")
events_data = create_sample_dataset()
sql_connection = setup_sql_database(events_data)

Setting up sample dataset and database...


In [66]:
print(f"Dataset created with {len(events_data)} events")
print("Sample data:")
print(events_data.head())
print("\nDataset info:")
print(events_data.info())
print("\nDevice type distribution:")
print(events_data['device_type'].value_counts())

Dataset created with 50000 events
Sample data:
   user_id                  timestamp device_type action_type  \
0     2111 2026-07-18 14:41:44.166352    smart_tv        view   
1     4289 2026-07-18 14:41:49.166352      mobile       click   
2     4115 2026-07-18 14:42:02.166352      mobile      search   
3     3349 2026-07-18 14:42:23.166352     desktop       click   
4     3999 2026-07-18 14:42:36.166352      mobile      search   

   session_duration_minutes  
0                 11.037380  
1                 24.894000  
2                  0.823399  
3                 11.885354  
4                  8.871059  

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   50000 non-null  int64         
 1   timestamp                 50000 non-null  datetime64[us]
 2   device_type      

In [67]:
events_data.head(5).T

,0,1,2,3,4
user_id,2111,4289,4115,3349,3999
timestamp,2026-07-18 14:41:44.166352,2026-07-18 14:41:49.166352,2026-07-18 14:42:02.166352,2026-07-18 14:42:23.166352,2026-07-18 14:42:36.166352
device_type,smart_tv,mobile,mobile,desktop,mobile
action_type,view,click,search,click,search
session_duration_minutes,11.03738,24.894,0.823399,11.885354,8.871059


In [68]:
# SQL DIALECT COMPARISON SECTION
print("\n" + "="*60)
print("SQL DIALECT COMPARISON")
print("="*60)


SQL DIALECT COMPARISON


In [69]:
# PROVIDED CODE - DO NOT MODIFY
# ANSI-SQL window function example
ansi_sql_query = """
SELECT user_id, device_type, session_duration_minutes,
       ROW_NUMBER() OVER (
           PARTITION BY device_type
           ORDER BY session_duration_minutes DESC
       ) AS session_rank
FROM user_events
WHERE action_type = 'view'
LIMIT 10;
"""

In [70]:
# Spark-SQL equivalent (syntax differences highlighted)
spark_sql_query = """
SELECT user_id, device_type, session_duration_minutes,
       ROW_NUMBER() OVER (
           PARTITION BY device_type
           ORDER BY session_duration_minutes DESC ROWS UNBOUNDED PRECEDING
       ) AS session_rank
FROM user_events
WHERE action_type = 'view'
LIMIT 10;
"""

In [71]:
print("ANSI-SQL Query:")
print(ansi_sql_query)
print("\nSpark-SQL Query:")
print(spark_sql_query)

ANSI-SQL Query:

SELECT user_id, device_type, session_duration_minutes,
       ROW_NUMBER() OVER (
           PARTITION BY device_type
           ORDER BY session_duration_minutes DESC
       ) AS session_rank
FROM user_events
WHERE action_type = 'view'
LIMIT 10;


Spark-SQL Query:

SELECT user_id, device_type, session_duration_minutes,
       ROW_NUMBER() OVER (
           PARTITION BY device_type
           ORDER BY session_duration_minutes DESC ROWS UNBOUNDED PRECEDING
       ) AS session_rank
FROM user_events
WHERE action_type = 'view'
LIMIT 10;



In [72]:
# Test both queries
print("ANSI-SQL Results:")
ansi_results = pd.read_sql_query(ansi_sql_query, sql_connection)
print(ansi_results)

ANSI-SQL Results:
   user_id device_type  session_duration_minutes  session_rank
0     3209     desktop                152.269642             1
1     2104     desktop                149.359314             2
2     1525     desktop                125.161769             3
3     2038     desktop                125.154054             4
4     3535     desktop                115.686514             5
5     2591     desktop                114.268888             6
6     2231     desktop                109.607116             7
7     4209     desktop                104.800781             8
8     2155     desktop                102.441361             9
9     3537     desktop                102.244937            10


In [73]:
print("\nNote: SQLite uses ANSI-SQL syntax. Spark-SQL differences would be visible in actual Spark environment.")


Note: SQLite uses ANSI-SQL syntax. Spark-SQL differences would be visible in actual Spark environment.


## EDA

In [74]:
import plotly.express as px

fig = px.pie(events_data, names='device_type', title='Events by Device Type')
fig.show()

In [75]:
action_counts = events_data['action_type'].value_counts().reset_index()
fig = px.bar(action_counts, x='action_type', y='count', title='Events by Action Type')
fig.show()

In [76]:
fig = px.histogram(events_data, x='session_duration_minutes', nbins=50,
                   title='Session Duration Distribution')
fig.show()

In [77]:
# hourly_counts = events_data.groupby('hour').size().reset_index(name='event_count')



# fig = px.line(hourly_counts, x='hour', y='event_count', title='Hourly Event Volume')
# fig.show()

### Session Duration by Device Type (Box Plot)

This chart compares how long sessions last across different device types (mobile, desktop, tablet, smart_tv).

- **Box** — the middle 50% of session durations (IQR)
- **Line inside box** — the median session duration
- **Whiskers** — the spread of typical values beyond the IQR
- **Dots** — outliers (unusually long sessions)

Use this to spot which devices tend to have longer or more variable sessions — e.g. smart TV users may watch longer than mobile users.

In [78]:
fig = px.box(events_data, x='device_type', y='session_duration_minutes',
             title='Session Duration by Device Type')
fig.show()

In [79]:
fig = px.histogram(events_data, x='device_type', color='action_type', barmode='group',
                   title='Action Types per Device')
fig.show()

PRACTICE CHALLENGE 1
TASK: Write equivalent window function queries in both ANSI-SQL and Spark-SQL syntax
to rank user sessions by duration within each device type.
Focus on the window frame specification differences between dialects.
YOUR CODE HERE
Hint: Create two query strings that rank sessions by duration within device types
Pay attention to how window frames are specified differently

In [80]:
events_data

,user_id,timestamp,device_type,action_type,session_duration_minutes
0,2111,2026-07-18 14:41:44.166352,smart_tv,view,11.037380
1,4289,2026-07-18 14:41:49.166352,mobile,click,24.894000
2,4115,2026-07-18 14:42:02.166352,mobile,search,0.823399
3,3349,2026-07-18 14:42:23.166352,desktop,click,11.885354
4,3999,2026-07-18 14:42:36.166352,mobile,search,8.871059
...,...,...,...,...,...
49995,3487,2026-07-25 14:40:56.166352,desktop,view,22.458072
49996,2375,2026-07-25 14:40:56.166352,desktop,search,23.871425
49997,4364,2026-07-25 14:40:58.166352,mobile,click,0.409538
49998,4512,2026-07-25 14:40:58.166352,tablet,search,1.930679


In [81]:
import pandas as pd

events_data['rank'] = (
    events_data.groupby(['user_id', 'device_type'])['session_duration_minutes']
      .rank(method='min', ascending=False)
      .astype(int)
)

result = events_data.sort_values('user_id')[
    ['user_id', 'device_type', 'session_duration_minutes', 'rank']
]

result = events_data[
    ['user_id', 'device_type', 'session_duration_minutes', 'rank']
].sort_values(['device_type', 'rank'])
result

,user_id,device_type,session_duration_minutes,rank
20,4477,desktop,16.648261,1
22,1327,desktop,27.760122,1
26,1667,desktop,16.558923,1
34,2611,desktop,63.500754,1
37,2976,desktop,37.248181,1
...,...,...,...,...
11568,2573,tablet,1.318383,9
15838,2604,tablet,1.893016,9
38638,4994,tablet,2.962160,9
1522,3964,tablet,1.675620,10


In [82]:
events_data.groupby(['user_id', 'session_duration_minutes']).agg({'session_duration_minutes': 'sum'})

session_duration_minutes
user_id session_duration_minutes                          
1000    1.925733                                  1.925733
        5.817295                                  5.817295
        6.301781                                  6.301781
        10.336373                                10.336373
        13.207710                                13.207710
...                                                    ...
4999    17.745983                                17.745983
        21.377649                                21.377649
        32.346436                                32.346436
        41.962557                                41.962557
        51.745235                                51.745235

[50000 rows x 1 columns]

In [83]:
ansi_ranking_query = """
SELECT
    user_id,
    device_type,
    session_duration_minutes,
    RANK() OVER (
        PARTITION BY device_type
        ORDER BY session_duration_minutes DESC
    ) AS duration_rank
FROM user_events
ORDER BY device_type, duration_rank
LIMIT 20;
"""

print("ANSI-SQL ranking query:")
print(ansi_ranking_query)
ansi_ranking_results = pd.read_sql_query(ansi_ranking_query, sql_connection)
print(ansi_ranking_results)


ANSI-SQL ranking query:

SELECT
    user_id,
    device_type,
    session_duration_minutes,
    RANK() OVER (
        PARTITION BY device_type
        ORDER BY session_duration_minutes DESC
    ) AS duration_rank
FROM user_events
ORDER BY device_type, duration_rank
LIMIT 20;

    user_id device_type  session_duration_minutes  duration_rank
0      3673     desktop                165.826292              1
1      3209     desktop                152.269642              2
2      2104     desktop                149.359314              3
3      4827     desktop                144.802487              4
4      1525     desktop                125.161769              5
5      2038     desktop                125.154054              6
6      2941     desktop                118.782472              7
7      3535     desktop                115.686514              8
8      2591     desktop                114.268888              9
9      4360     desktop                113.098781             10
10     2

In [84]:
spark_ranking_query = """
SELECT
    user_id,
    device_type,
    session_duration_minutes,
    RANK() OVER (
        PARTITION BY device_type
        ORDER BY session_duration_minutes DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS duration_rank
FROM user_events
ORDER BY device_type, duration_rank
LIMIT 20;
"""

# Note: Spark-SQL requires explicit window frame specification.
# ANSI-SQL allows the frame to be implicit for RANK(); Spark does not.
print("Spark-SQL ranking query (for reference — runs in Spark, not SQLite):")
print(spark_ranking_query)


Spark-SQL ranking query (for reference — runs in Spark, not SQLite):

SELECT
    user_id,
    device_type,
    session_duration_minutes,
    RANK() OVER (
        PARTITION BY device_type
        ORDER BY session_duration_minutes DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS duration_rank
FROM user_events
ORDER BY device_type, duration_rank
LIMIT 20;



Test your queries (we'll use the ANSI version since we're using SQLite)

In [85]:
print("\n" + "="*60)
print("PANDAS AGGREGATION SECTION")
print("="*60)


PANDAS AGGREGATION SECTION


In [86]:
# PROVIDED CODE - DO NOT MODIFY
# Prepare datetime column for time-based aggregation
events_data['timestamp'] = pd.to_datetime(events_data['timestamp'])
events_data['hour'] = events_data['timestamp'].dt.floor('h')  # Round down to nearest hour

In [87]:
events_data

,user_id,timestamp,device_type,action_type,session_duration_minutes,rank,hour
0,2111,2026-07-18 14:41:44.166352,smart_tv,view,11.037380,2,2026-07-18 14:00:00
1,4289,2026-07-18 14:41:49.166352,mobile,click,24.894000,1,2026-07-18 14:00:00
2,4115,2026-07-18 14:42:02.166352,mobile,search,0.823399,8,2026-07-18 14:00:00
3,3349,2026-07-18 14:42:23.166352,desktop,click,11.885354,3,2026-07-18 14:00:00
4,3999,2026-07-18 14:42:36.166352,mobile,search,8.871059,6,2026-07-18 14:00:00
...,...,...,...,...,...,...,...
49995,3487,2026-07-25 14:40:56.166352,desktop,view,22.458072,1,2026-07-25 14:00:00
49996,2375,2026-07-25 14:40:56.166352,desktop,search,23.871425,2,2026-07-25 14:00:00
49997,4364,2026-07-25 14:40:58.166352,mobile,click,0.409538,6,2026-07-25 14:00:00
49998,4512,2026-07-25 14:40:58.166352,tablet,search,1.930679,5,2026-07-25 14:00:00


In [88]:
# PRACTICE CHALLENGE 2
# TASK: Create a comprehensive hourly aggregation that includes:
# - session_count: Number of sessions per hour
# - unique_users: Number of unique users per hour
# - avg_session_duration: Average session duration per hour
# - most_popular_device: Most common device type per hour
# Hint: Combine multiple aggregation functions in a single groupby operation for efficiency
# YOUR CODE HERE
# hourly_metrics = None  # Replace with your aggregation logic
# Uncomment and complete:
hourly_metrics = (
    events_data.groupby('hour')
    .agg(
        session_count=('user_id', 'count'),
        unique_users=('user_id', 'nunique'),
        avg_session_duration=('session_duration_minutes', 'mean'),
        most_popular_device=('device_type', lambda x: x.mode()[0])
    )
    .reset_index()
)
hourly_metrics
# YOUR AGGREGATION FUNCTIONS HERE
# })

,hour,session_count,unique_users,avg_session_duration,most_popular_device
0,2026-07-18 14:00:00,83,81,14.031598,mobile
1,2026-07-18 15:00:00,296,283,15.687886,mobile
2,2026-07-18 16:00:00,295,289,14.562448,mobile
3,2026-07-18 17:00:00,297,284,15.940238,mobile
4,2026-07-18 18:00:00,308,302,14.423512,mobile
...,...,...,...,...,...
164,2026-07-25 10:00:00,280,272,15.810833,mobile
165,2026-07-25 11:00:00,295,283,13.800881,mobile
166,2026-07-25 12:00:00,301,293,14.471415,mobile
167,2026-07-25 13:00:00,309,299,14.465610,mobile


In [89]:
print("\n" + "="*60)
print("DATA PIPELINE OPTIMIZATION SECTION")
print("="*60)


DATA PIPELINE OPTIMIZATION SECTION


In [90]:
# PRACTICE CHALLENGE 3
# TASK: Build a complete data pipeline function that processes event data and
# exports optimized Parquet files for downstream analytics.
# The function should:
# 1. Accept a DataFrame of raw events
# 2. Perform time-based aggregations
# 3. Optimize data types for memory efficiency
# 4. Export results to Parquet format
# 5. Return summary statistics about the processing
# Hint: Consider memory optimization and data type efficiency in your implementation
# YOUR CODE HERE
import pandas as pd
import numpy as np
from pathlib import Path


def create_analytics_pipeline(events_df, output_filename='hourly_metrics.parquet'):
    """
    Complete data pipeline for event aggregation and export

    Args:
        events_df (pd.DataFrame): Raw event data
        output_filename (str): Output file path for Parquet export

    Returns:
        dict: Summary statistics about the processing
    """
    df = events_df.copy()
    mem_before = df.memory_usage(deep=True).sum() / 1024**2  # MB

    # --- Ensure timestamp is datetime, floor to hour ---
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['hour'] = df['timestamp'].dt.floor('h')

    # --- Time-based aggregation ---
    hourly_metrics = (
        df.groupby('hour')
        .agg(
            session_count=('user_id', 'count'),
            unique_users=('user_id', 'nunique'),
            avg_session_duration=('session_duration_minutes', 'mean'),
            most_popular_device=('device_type', lambda x: x.mode()[0])
        )
        .reset_index()
    )
    hourly_metrics['avg_session_duration'] = hourly_metrics['avg_session_duration'].round(2)

    # --- Optimize dtypes for memory efficiency ---
    def optimize_dtypes(data):
        data = data.copy()
        for col in data.select_dtypes(include=['int64']).columns:
            data[col] = pd.to_numeric(data[col], downcast='integer')
        for col in data.select_dtypes(include=['float64']).columns:
            data[col] = pd.to_numeric(data[col], downcast='float')
        for col in data.select_dtypes(include=['object']).columns:
            n_unique = data[col].nunique()
            n_total = len(data[col])
            if n_total > 0 and n_unique / n_total < 0.5:
                data[col] = data[col].astype('category')
        return data

    hourly_metrics = optimize_dtypes(hourly_metrics)
    mem_after = df.memory_usage(deep=True).sum() / 1024**2  # MB (raw df, unchanged)

    # --- Export to Parquet ---
    output_path = Path(output_filename)
    hourly_metrics.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)

    # --- Summary statistics ---
    summary = {
        'rows_processed': len(df),
        'hours_aggregated': len(hourly_metrics),
        'unique_users_total': df['user_id'].nunique(),
        'date_range_start': str(df['hour'].min()),
        'date_range_end': str(df['hour'].max()),
        'output_file': str(output_path),
        'output_file_size_mb': round(output_path.stat().st_size / 1024**2, 2),  
        'memory_before_mb': round(mem_before, 2),
        'memory_after_mb': round(mem_after, 2)  
    }

    return summary


# --- Run it ---
summary = create_analytics_pipeline(events_data)
summary

{'rows_processed': 50000,
 'hours_aggregated': 169,
 'unique_users_total': 4000,
 'date_range_start': '2026-07-18 14:00:00',
 'date_range_end': '2026-07-25 14:00:00',
 'output_file': 'hourly_metrics.parquet',
 'output_file_size_mb': 0.01,
 'memory_before_mb': np.float64(3.22),
 'memory_after_mb': np.float64(3.22)}

In [91]:
# Test your pipeline
pipeline_results = create_analytics_pipeline(events_data)
print("Pipeline results:", pipeline_results)

Pipeline results: {'rows_processed': 50000, 'hours_aggregated': 169, 'unique_users_total': 4000, 'date_range_start': '2026-07-18 14:00:00', 'date_range_end': '2026-07-25 14:00:00', 'output_file': 'hourly_metrics.parquet', 'output_file_size_mb': 0.01, 'memory_before_mb': np.float64(3.22), 'memory_after_mb': np.float64(3.22)}


In [92]:
print("\n" + "="*60)
print("TESTING AND VALIDATION")
print("="*60)


TESTING AND VALIDATION


In [96]:
# PROVIDED CODE - DO NOT MODIFY
def validate_results():
    """Validation function to check if all challenges are completed correctly"""
    print("Validation checklist:")

    # Check if SQL queries are defined
    try:
        if 'ansi_ranking_query' in globals() and len(ansi_ranking_query.strip()) > 50:
            print("✅ ANSI-SQL ranking query created")
        else:
            print("❌ ANSI-SQL ranking query needs completion")

        if 'spark_ranking_query' in globals() and len(spark_ranking_query.strip()) > 50:
            print("✅ Spark-SQL ranking query created")
        else:
            print("❌ Spark-SQL ranking query needs completion")
    except:
        print("❌ SQL queries need to be defined")

    # Check if hourly metrics are created
    try:
        if hourly_metrics is not None and len(hourly_metrics) > 0:
            print("✅ Hourly metrics aggregation completed")
            print(f"✅ Generated {len(hourly_metrics)} hourly data points")
        else:
            print("❌ Hourly metrics aggregation needs completion")
    except:
        print("❌ Hourly metrics variable needs to be defined")

    # Check if pipeline function is implemented
    try:
        if hasattr(create_analytics_pipeline, '__code__') and len(create_analytics_pipeline.__code__.co_names) > 3:
            print("✅ Analytics pipeline function implemented")
        else:
            print("❌ Analytics pipeline function needs implementation")
    except:
        print("❌ Analytics pipeline function needs work")

In [97]:
# Run validation
validate_results()

Validation checklist:
✅ ANSI-SQL ranking query created
✅ Spark-SQL ranking query created
✅ Hourly metrics aggregation completed
✅ Generated 169 hourly data points
✅ Analytics pipeline function implemented


In [99]:
print("\n" + "="*60)
print("LAB COMPLETE!")
print("Remember to check your work against the success checklist in the instructions.")
print("="*60)


LAB COMPLETE!
Remember to check your work against the success checklist in the instructions.
